# BDC Satria Data 2026 — 3 Model Perbaikan (DINOv2-Base, RegNetY-16GF, DenseNet-201)

**Ini versi khusus cuma 3 model yang GAGAL di run sebelumnya, sudah diperbaiki:**

1. **DINOv2 ViT-Base** — `vit_base_patch14_dinov2.lvd142m`, ~86.6M params. FIX: tambah `dynamic_img_size=True` (native resolusinya 518, bukan 224 -- ini yg bikin error `Input height (224) doesn't match model (518)` sebelumnya).

2. **RegNetY-16GF** — `regnety_160.tv2_in1k` (~83M params). FIX: dipindah dari `torchvision` ke `timm`. Sebelumnya gagal `[ASN1: NOT_ENOUGH_DATA]` SSL error saat download dari `pytorch.org` -- sekarang download lewat HuggingFace Hub (host yang sama dengan model kamu yang lain, sudah terbukti jalan).

3. **DenseNet-201** (bukan 264) — `densenet201.tv_in1k`, ~20M params. **DenseNet-264 dicek langsung ke timm dan TIDAK ADA pretrained weights sama sekali** untuk kedalaman itu, di varian manapun (`densenet264`, `densenet264d`, dst). DenseNet-201 adalah yang terdalam yang tersedia dengan bobot ImageNet -- catatan penting: ini model jauh lebih kecil (~20M vs perkiraan awal ~73M utk 264), jadi hasilnya mungkin tidak representatif untuk "DenseNet besar", tapi ini pilihan terbaik yang benar-benar tersedia.

3 model yang SUDAH SUKSES sebelumnya (DeiT-Base 0.9513, CoAtNet-2 0.9698, PVTv2-B5 0.9535) **dihapus dari `MODEL_LIST`** di notebook ini -- tidak perlu training ulang.

**Konsisten dgn notebook lain:** `StratifiedShuffleSplit` 3-fold 90/10, class weight balanced, macro F1, tracking misclassified → `.xlsx` 2 sheet, try/except per model (kalau masih ada yang gagal, notebook tetap lanjut ke model berikutnya).

## 1. Setup & Konfigurasi

In [2]:
from __future__ import annotations
import os
import gc
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import models as tv_models, transforms as T
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image

try:
    import timm
except ImportError:
    raise ImportError("timm belum terinstall. Jalankan: pip install timm")

In [ ]:
# === PATH DATASET ===
DATA_ROOT = Path(r"C:\Users\MyPC PRO\Downloads\BDC2026")
TRAIN_DIR = DATA_ROOT / "train"
MODELS_DIR = DATA_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = DATA_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

CLASS_FOLDERS = ["0_Recyclable", "1_Electronic", "2_Organic"]
CLASS_NAMES = ["Recyclable", "Electronic", "Organic"]
NUM_CLASSES = len(CLASS_NAMES)

# === MODEL LIST ===
MODEL_LIST = [
    {"key": "dinov2_base",     "type": "timm", "timm_name": "vit_base_patch14_dinov2.lvd142m",        "img_size": 224, "dynamic_img_size": True},  # native res 518, bukan 224
    {"key": "regnety_16gf",    "type": "timm", "timm_name": "regnety_160.tv2_in1k",                   "img_size": 224},  # FIX: pindah dari torchvision ke timm (hindari SSL error pytorch.org, download via HF Hub)
    {"key": "densenet201",     "type": "timm", "timm_name": "densenet201.tv_in1k",                     "img_size": 224},  # FIX: DenseNet-264 TIDAK ADA pretrained weights di timm sama sekali -- diganti DenseNet-201 (terdalam yang tersedia)
]

# === CONFIG TRAINING (RTX 3060 12.9GB) ===
BATCH_SIZE = 128          # seragam utk semua 6 model (semua kelas ~75-90M @ res 224)
GRAD_ACCUM_STEPS = 1
NUM_EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 4
N_FOLDS = 3
FOLD_VAL_FRAC = 0.10
SEED = 42
NUM_WORKERS = 8

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Jumlah model dalam antrian: {len(MODEL_LIST)}")
for m in MODEL_LIST:
    print(f"  - {m['key']}")

Device: cuda
GPU: NVIDIA GeForce RTX 3060
VRAM: 12.9 GB
Jumlah model dalam antrian: 3
  - dinov2_base
  - regnety_16gf
  - densenet201


## 2. Index Dataset dari Folder

In [4]:
def index_dataset(train_dir: Path, class_folders: list[str]):
    records = []
    for label_idx, folder_name in enumerate(class_folders):
        folder = train_dir / folder_name
        if not folder.is_dir():
            raise FileNotFoundError(f"Folder tidak ditemukan: {folder}")
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        files = [p for p in folder.iterdir() if p.suffix.lower() in exts]
        for p in files:
            records.append({"path": str(p), "label": label_idx, "class_name": CLASS_NAMES[label_idx]})
    return pd.DataFrame(records)

full_df = index_dataset(TRAIN_DIR, CLASS_FOLDERS)
print(f"Total citra ter-index: {len(full_df)}")
print(full_df["class_name"].value_counts())

Total citra ter-index: 26527
class_name
Organic       12567
Recyclable     9999
Electronic     3961
Name: count, dtype: int64


## 3. Dataset & Transform

In [5]:
# TrashDataset & build_transforms dipindah ke dataset_utils_six.py (wajib jadi modul .py asli
# biar DataLoader num_workers>0 gak hang di Windows/Jupyter - spawn worker perlu import modul nyata)
from dataset_utils_six import TrashDataset, build_transforms

## 4. Model Builder (generik: timm ATAU torchvision RegNet)

In [6]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def build_model(model_cfg: dict, num_classes: int):
    if model_cfg["type"] == "timm":
        # dynamic_img_size=True: perlu utk model yg native resolution-nya beda dari IMG_SIZE kita (224),
        # spt DINOv2-Base (native 518). Aman dipasang utk semua model timm di sini, tdk berpengaruh
        # ke model yg native-nya memang sudah 224.
        create_kwargs = {"pretrained": True, "num_classes": num_classes}
        if model_cfg.get("dynamic_img_size"):
            # cuma model ViT-style yang dukung param ini (mis. DINOv2 native res 518) - CNN (RegNetY/DenseNet)
            # bakal TypeError kalau dipaksa dikasih dynamic_img_size
            create_kwargs["dynamic_img_size"] = True
        model = timm.create_model(model_cfg["timm_name"], **create_kwargs)
        for p in model.parameters():
            p.requires_grad = False
        head_module = None
        for attr in ["head", "classifier", "fc"]:
            if hasattr(model, attr):
                candidate = getattr(model, attr)
                if isinstance(candidate, nn.Module):
                    head_module = candidate
                    break
        if head_module is None:
            raise AttributeError(f"Head classifier tidak ketemu untuk {model_cfg['timm_name']}")
        for p in head_module.parameters():
            p.requires_grad = True
        data_cfg = timm.data.resolve_model_data_config(model)
        mean, std = data_cfg["mean"], data_cfg["std"]

    else:
        raise ValueError(f"Tipe model tidak dikenal: {model_cfg['type']}")

    return model.to(DEVICE), mean, std

## 5. Train & Evaluate

In [7]:
scaler = GradScaler("cuda", enabled=(DEVICE == "cuda"))


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    optimizer.zero_grad()

    for step, (imgs, labels, _) in enumerate(loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels) / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * GRAD_ACCUM_STEPS
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / len(loader), macro_f1


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_paths, all_probs = [], [], [], []
    for imgs, labels, paths in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        running_loss += loss.item()

        probs = F.softmax(outputs.float(), dim=1)
        preds = probs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_paths.extend(paths)
        all_probs.extend(probs.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / len(loader), macro_f1, np.array(all_labels), np.array(all_preds), all_paths, np.array(all_probs)

## 6. Loop Utama — 6 Model x 3 Fold, dengan Try/Except per Model

Kalau 1 model gagal (misal nama checkpoint salah), dicatat di `failed_models` dan notebook lanjut ke model berikutnya — tidak berhenti total.

In [ ]:
all_fold_results = []
detail_records = []
eval_counter = defaultdict(int)
failed_models = []

for model_cfg in MODEL_LIST:
    model_key = model_cfg["key"]
    img_size = model_cfg["img_size"]
    print(f"\n{'#'*70}\n### MODEL: {model_key}\n{'#'*70}")

    try:
        sss = StratifiedShuffleSplit(n_splits=N_FOLDS, test_size=FOLD_VAL_FRAC, random_state=SEED)

        for fold, (train_idx, val_idx) in enumerate(sss.split(full_df, full_df["label"]), start=1):
            print(f"\n{'='*64}\n---> [{model_key}] FOLD {fold}/{N_FOLDS} <---\n{'='*64}")
            train_df = full_df.iloc[train_idx]
            val_df = full_df.iloc[val_idx]

            weights = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_df["label"].values)
            class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
            print(f"  Class weights: {dict(zip(CLASS_NAMES, weights.round(3)))}")

            model, mean, std = build_model(model_cfg, NUM_CLASSES)
            if fold == 1:
                n_total = sum(p.numel() for p in model.parameters())
                n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
                print(f"  Total params: {n_total:,} | Trainable (head): {n_train:,}")

            train_tf = build_transforms(mean, std, img_size, train=True)
            val_tf = build_transforms(mean, std, img_size, train=False)

            train_loader = DataLoader(TrashDataset(train_df, train_tf),
                                      batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
                                      persistent_workers=(NUM_WORKERS > 0), prefetch_factor=4 if NUM_WORKERS > 0 else None)
            val_loader = DataLoader(TrashDataset(val_df, val_tf),
                                    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
                                    persistent_workers=(NUM_WORKERS > 0), prefetch_factor=4 if NUM_WORKERS > 0 else None)

            criterion = nn.CrossEntropyLoss(weight=class_weights)
            optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                          lr=LR, weight_decay=WEIGHT_DECAY)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

            best_f1 = 0.0
            epochs_no_improve = 0
            best_path = MODELS_DIR / f"{model_key}_fold{fold}.pth"

            for epoch in range(1, NUM_EPOCHS + 1):
                tr_loss, tr_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
                val_loss, val_f1, _, _, _, _ = evaluate(model, val_loader, criterion)
                scheduler.step(val_f1)

                marker = ""
                if val_f1 > best_f1:
                    best_f1 = val_f1
                    epochs_no_improve = 0
                    torch.save(model.state_dict(), best_path)
                    marker = " [*] best"
                else:
                    epochs_no_improve += 1

                print(f"  Epoch {epoch:2d}/{NUM_EPOCHS} | train_loss {tr_loss:.4f} F1 {tr_f1:.4f} | "
                      f"val_loss {val_loss:.4f} F1 {val_f1:.4f}{marker}")

                if epochs_no_improve >= PATIENCE:
                    print(f"  Early stopping.")
                    break

            model.load_state_dict(torch.load(best_path, weights_only=True))
            _, final_f1, y_true, y_pred, paths, probs = evaluate(model, val_loader, criterion)
            print(f"  >> [{model_key}] Fold {fold} best macro F1: {best_f1:.4f} | re-eval F1: {final_f1:.4f}")

            all_fold_results.append({"model": model_key, "fold": fold, "best_macro_f1": best_f1})

            for i, path in enumerate(paths):
                eval_counter[(model_key, path)] += 1
                if y_true[i] != y_pred[i]:
                    p = probs[i]
                    detail_records.append({
                        "model": model_key, "fold": fold, "filepath": path,
                        "true_label": CLASS_NAMES[y_true[i]],
                        "predicted_label": CLASS_NAMES[y_pred[i]],
                        "confidence_pct": round(float(p[y_pred[i]]) * 100, 2),
                        "true_label_confidence_pct": round(float(p[y_true[i]]) * 100, 2),
                        "prob_recyclable_pct": round(float(p[0]) * 100, 2),
                        "prob_electronic_pct": round(float(p[1]) * 100, 2),
                        "prob_organic_pct": round(float(p[2]) * 100, 2),
                    })

            del model, train_loader, val_loader, optimizer, scheduler
            gc.collect()
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"\n  [GAGAL] Model {model_key} error: {e}")
        print(f"  Melanjutkan ke model berikutnya...")
        failed_models.append({"model": model_key, "error": str(e)})
        torch.cuda.empty_cache()
        continue

print(f"\n\n{'='*70}\nSemua model selesai diproses. Model gagal: {len(failed_models)}/{len(MODEL_LIST)}\n{'='*70}")
if failed_models:
    for f in failed_models:
        print(f"  - {f['model']}: {f['error']}")


######################################################################
### MODEL: dinov2_base
######################################################################

---> [dinov2_base] FOLD 1/3 <---
  Class weights: {'Recyclable': np.float64(0.884), 'Electronic': np.float64(2.232), 'Organic': np.float64(0.704)}
  Total params: 86,582,019 | Trainable (head): 2,307


## 6b. Retry khusus model yang tadi gagal (regnety_16gf, densenet201)
Dipisah biar gak perlu re-run `dinov2_base` yang sudah berhasil. Pakai variable akumulator yang sama (`all_fold_results`, `detail_records`, `eval_counter`, `failed_models`) supaya ringkasan & laporan xlsx di bawah otomatis include hasil retry ini.

In [ ]:
RETRY_KEYS = ["regnety_16gf", "densenet201"]
retry_models = [m for m in MODEL_LIST if m["key"] in RETRY_KEYS]

# buang hasil gagal yang lama biar summary/failed_models gak nyangkut duplikat dari attempt sebelumnya
failed_models = [f for f in failed_models if f["model"] not in RETRY_KEYS]

for model_cfg in retry_models:
    model_key = model_cfg["key"]
    img_size = model_cfg["img_size"]
    print(f"\n{'#'*70}\n### MODEL (retry): {model_key}\n{'#'*70}")

    try:
        sss = StratifiedShuffleSplit(n_splits=N_FOLDS, test_size=FOLD_VAL_FRAC, random_state=SEED)

        for fold, (train_idx, val_idx) in enumerate(sss.split(full_df, full_df["label"]), start=1):
            print(f"\n{'='*64}\n---> [{model_key}] FOLD {fold}/{N_FOLDS} <---\n{'='*64}")
            train_df = full_df.iloc[train_idx]
            val_df = full_df.iloc[val_idx]

            weights = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_df["label"].values)
            class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
            print(f"  Class weights: {dict(zip(CLASS_NAMES, weights.round(3)))}")

            model, mean, std = build_model(model_cfg, NUM_CLASSES)
            if fold == 1:
                n_total = sum(p.numel() for p in model.parameters())
                n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
                print(f"  Total params: {n_total:,} | Trainable (head): {n_train:,}")

            train_tf = build_transforms(mean, std, img_size, train=True)
            val_tf = build_transforms(mean, std, img_size, train=False)

            train_loader = DataLoader(TrashDataset(train_df, train_tf),
                                      batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
                                      persistent_workers=(NUM_WORKERS > 0), prefetch_factor=4 if NUM_WORKERS > 0 else None)
            val_loader = DataLoader(TrashDataset(val_df, val_tf),
                                    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
                                    persistent_workers=(NUM_WORKERS > 0), prefetch_factor=4 if NUM_WORKERS > 0 else None)

            criterion = nn.CrossEntropyLoss(weight=class_weights)
            optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                          lr=LR, weight_decay=WEIGHT_DECAY)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

            best_f1 = 0.0
            epochs_no_improve = 0
            best_path = MODELS_DIR / f"{model_key}_fold{fold}.pth"

            for epoch in range(1, NUM_EPOCHS + 1):
                tr_loss, tr_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
                val_loss, val_f1, _, _, _, _ = evaluate(model, val_loader, criterion)
                scheduler.step(val_f1)

                marker = ""
                if val_f1 > best_f1:
                    best_f1 = val_f1
                    epochs_no_improve = 0
                    torch.save(model.state_dict(), best_path)
                    marker = " [*] best"
                else:
                    epochs_no_improve += 1

                print(f"  Epoch {epoch:2d}/{NUM_EPOCHS} | train_loss {tr_loss:.4f} F1 {tr_f1:.4f} | "
                      f"val_loss {val_loss:.4f} F1 {val_f1:.4f}{marker}")

                if epochs_no_improve >= PATIENCE:
                    print(f"  Early stopping.")
                    break

            model.load_state_dict(torch.load(best_path, weights_only=True))
            _, final_f1, y_true, y_pred, paths, probs = evaluate(model, val_loader, criterion)
            print(f"  >> [{model_key}] Fold {fold} best macro F1: {best_f1:.4f} | re-eval F1: {final_f1:.4f}")

            all_fold_results.append({"model": model_key, "fold": fold, "best_macro_f1": best_f1})

            for i, path in enumerate(paths):
                eval_counter[(model_key, path)] += 1
                if y_true[i] != y_pred[i]:
                    p = probs[i]
                    detail_records.append({
                        "model": model_key, "fold": fold, "filepath": path,
                        "true_label": CLASS_NAMES[y_true[i]],
                        "predicted_label": CLASS_NAMES[y_pred[i]],
                        "confidence_pct": round(float(p[y_pred[i]]) * 100, 2),
                        "true_label_confidence_pct": round(float(p[y_true[i]]) * 100, 2),
                        "prob_recyclable_pct": round(float(p[0]) * 100, 2),
                        "prob_electronic_pct": round(float(p[1]) * 100, 2),
                        "prob_organic_pct": round(float(p[2]) * 100, 2),
                    })

            del model, train_loader, val_loader, optimizer, scheduler
            gc.collect()
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"\n  [GAGAL] Model {model_key} error: {e}")
        print(f"  Melanjutkan ke model berikutnya...")
        failed_models.append({"model": model_key, "error": str(e)})
        torch.cuda.empty_cache()
        continue

print(f"\n\n{'='*70}\nRetry selesai. Model gagal (retry): {sum(1 for f in failed_models if f['model'] in RETRY_KEYS)}/{len(RETRY_KEYS)}\n{'='*70}")

## 7. Ringkasan Hasil — Perbandingan Semua Model

In [ ]:
results_df = pd.DataFrame(all_fold_results)
print("Ringkasan 3-Fold CV per model (macro F1):\n")
for model_key in results_df["model"].unique():
    sub = results_df[results_df["model"] == model_key]
    f1s = sub["best_macro_f1"].values
    print(f"  {model_key}:")
    for _, row in sub.iterrows():
        print(f"    Fold {int(row['fold'])}: {row['best_macro_f1']:.4f}")
    print(f"    Rata-rata: {np.mean(f1s):.4f} (+/- {np.std(f1s):.4f})\n")

print("Pembanding model kamu sebelumnya:")
print("  ViT-B/16 SWAG        : 0.9636")
print("  EVA-02 Base          : 0.9646")
print("  BEiT-Base            : 0.9695")
print("  ConvNeXt V2-Large    : 0.9720")
print("  ViT-L/16 SWAG        : 0.9770")
print("  SwinV2-Base          : 0.9776  <- terbaik sejauh ini")

results_df.to_csv(OUTPUT_DIR / "six_new_models_fold_results.csv", index=False)
if failed_models:
    pd.DataFrame(failed_models).to_csv(OUTPUT_DIR / "six_new_models_failed.csv", index=False)

## 8. Laporan Misclassified (`.xlsx`, 2 sheet, kolom `model`)

In [ ]:
detail_df = pd.DataFrame(detail_records, columns=[
    "model", "fold", "filepath", "true_label", "predicted_label",
    "confidence_pct", "true_label_confidence_pct",
    "prob_recyclable_pct", "prob_electronic_pct", "prob_organic_pct",
])

summary_rows = []
if len(detail_df) > 0:
    for (model_key, path), g in detail_df.groupby(["model", "filepath"]):
        folds_wrong = sorted(g["fold"].unique().tolist())
        total_eval = eval_counter[(model_key, path)]
        summary_rows.append({
            "model": model_key,
            "filepath": path,
            "true_label": g["true_label"].iloc[0],
            "folds_wrong": ",".join(map(str, folds_wrong)),
            "total_wrong": len(g),
            "total_evaluated": total_eval,
            "error_rate": round(len(g) / total_eval, 3) if total_eval else None,
            "avg_confidence_pct": round(g["confidence_pct"].mean(), 2),
            "avg_true_label_confidence_pct": round(g["true_label_confidence_pct"].mean(), 2),
            "avg_prob_recyclable_pct": round(g["prob_recyclable_pct"].mean(), 2),
            "avg_prob_electronic_pct": round(g["prob_electronic_pct"].mean(), 2),
            "avg_prob_organic_pct": round(g["prob_organic_pct"].mean(), 2),
        })

summary_df = pd.DataFrame(summary_rows, columns=[
    "model", "filepath", "true_label", "folds_wrong", "total_wrong", "total_evaluated", "error_rate",
    "avg_confidence_pct", "avg_true_label_confidence_pct",
    "avg_prob_recyclable_pct", "avg_prob_electronic_pct", "avg_prob_organic_pct",
])
if len(summary_df) > 0:
    summary_df = summary_df.sort_values(["error_rate", "total_wrong"], ascending=False).reset_index(drop=True)

report_path = OUTPUT_DIR / "misclassified_report_six_new_models.xlsx"
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    detail_df.to_excel(writer, sheet_name="Detail", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print(f"[SAVED] {report_path}")
print(f"  Detail  : {len(detail_df)} baris")
print(f"  Summary : {len(summary_df)} gambar unik")

## Catatan
- Kalau salah satu model OOM: turunkan `BATCH_SIZE` global, atau edit `MODEL_LIST` untuk jalankan subset saja (hapus entry yang bermasalah, run ulang khusus model itu dengan batch lebih kecil).
- Kalau `densenet264d` atau `coatnet_2_rw_224.sw_in12k` gagal load (dicatat di `failed_models`), berarti nama checkpoint pretrained perlu dicek ulang -- bisa saya bantu cari nama alternatif yang benar.
- Total waktu training 6 model x 3 fold ini kemungkinan panjang (bisa 6-12+ jam tergantung kecepatan tiap model). Pertimbangkan jalankan semalaman atau saat komputer kampus tidak dipakai.